# W06 — Validation and Research Claim Audit

**Lane:** Content-Day Performance / Decline Risk.

This notebook audits the Week-5 model rather than creating a new target.

The target is a proxy: a ≥20% decrease in GSC clicks per measured day from March 1–15 to March 16–31.

The headline evaluation uses a grouped client holdout. This tests whether the model generalizes to clients that were not included in training.

The notebook covers:

1. Research paper findings and methodology questions.
2. Validation design and available model results.
3. Data leakage audit.
4. Evidence-based claims and limitations.
5. False-positive and false-negative analysis.

**Important:** The Week-5 metrics file does not contain the random-row split diagnostic. We will report this limitation honestly rather than inventing results.

## 1. Two Paper Findings and Methodology Questions

### Finding 1 — [Insert the paper's exact finding]

- **What the paper reports:** [State the finding accurately and cite the section, table, or figure.]
- **Constructive methodology question:** How was the outcome or label defined? Was it measured independently of the model inputs? Could any feature contain information derived from the label?

### Finding 2 — [Insert the paper's exact finding]

- **What the paper reports:** [State the finding accurately and cite the section, table, or figure.]
- **Constructive methodology question:** What validation design was used? Does the split prevent related clients, content, or future observations from appearing on both sides? Does it support the scope of the paper's claims?

Keep the tone respectful. These questions are intended to understand the evidence, not make accusations.

**Submission requirement:** Replace both placeholders using the actual research paper assigned in the internship.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

# --------------------------------------------------
# 1. Locate repository and output directory
# --------------------------------------------------

REPO_ROOT = Path.cwd()

if not (REPO_ROOT / "work").exists():
    REPO_ROOT = Path("/content/FlyRank_Internship")

OUTPUT_DIR = REPO_ROOT / "work" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 2. Load Week-5 metrics
# --------------------------------------------------

metrics_path = OUTPUT_DIR / "w05_metrics.json"

assert metrics_path.exists(), (
    "w05_metrics.json was not found. "
    "Upload it into work/outputs."
)

metrics = json.loads(metrics_path.read_text())

# --------------------------------------------------
# 3. Display model comparison
# --------------------------------------------------

comparison = pd.DataFrame(metrics["comparison"])

print("WEEK 5 MODEL COMPARISON")
display(comparison)

# --------------------------------------------------
# 4. Load optional random split diagnostic
# --------------------------------------------------

random_diag = metrics.get("random_split_logistic_diagnostic")

if random_diag is not None:
    print("Random split diagnostic:")
    print(random_diag)
else:
    print(
        "Random split diagnostic is unavailable "
        "in the Week-5 metrics file."
    )

# --------------------------------------------------
# 5. Display grouped split information
# --------------------------------------------------

print("\nGrouped split information")
print("Training clients:", metrics["train_clients"])
print("Testing clients:", metrics["test_clients"])
print("Training rows:", metrics["train_rows"])
print("Testing rows:", metrics["test_rows"])
print("Base rate:", metrics["test_base_rate"])

## 2. Validation Design: Random Split vs. Grouped Split

### Random-row split

A random-row split distributes individual observations between training and testing. If multiple observations belong to the same client, that client may appear in both partitions. This can create an optimistic evaluation because the model may benefit from patterns associated with clients it has already encountered.

### Grouped-client holdout

The grouped split assigns entire clients to either training or testing. The same client cannot appear in both partitions. This provides a more realistic test of generalization to previously unseen clients.

### Available evidence

The Week-5 metrics file contains the grouped-client holdout results. However, it does not contain the random-row diagnostic metrics. Therefore, a numerical before-and-after comparison cannot be produced from the saved metrics alone.

The grouped-client holdout remains the headline evaluation.

In [ ]:
# --------------------------------------------------
# Grouped-client model results
# --------------------------------------------------

grouped_results = comparison[
    comparison["method"] == "Logistic Regression"
].copy()

assert not grouped_results.empty, (
    "Logistic Regression results are missing."
)

print("Grouped Client Holdout — Logistic Regression")
display(grouped_results)

grouped_results.to_csv(
    OUTPUT_DIR / "w06_grouped_results.csv",
    index=False
)

# --------------------------------------------------
# Random split comparison, only if saved in Week 5
# --------------------------------------------------

if random_diag is None:
    print(
        "\nRandom-row split comparison skipped. "
        "Week-5 metrics did not save the random-split diagnostic."
    )
else:
    selected_columns = [
        "base_rate",
        "average_precision",
        "roc_auc",
        "precision_at_10",
        "precision_at_20",
        "precision_at_50"
    ]

    random_row = {
        "design": "Random row split",
        **{
            key: random_diag[key]
            for key in selected_columns
            if key in random_diag
        }
    }

    grouped_row_data = grouped_results.iloc[0].to_dict()
    grouped_row = {"design": "Grouped client holdout"}

    for key in selected_columns:
        if key in grouped_row_data:
            grouped_row[key] = grouped_row_data[key]

    before_after = pd.DataFrame([random_row, grouped_row])
    display(before_after)

    before_after.to_csv(
        OUTPUT_DIR / "w06_split_comparison.csv",
        index=False
    )

## 3. Data Leakage Audit

Data leakage occurs when information unavailable at prediction time is included in the model's training features.

For this project:

- The feature window covers March 1–15.
- The outcome window covers March 16–31.
- The prediction cutoff is March 16.
- Later-window clicks are used to construct the target label only.
- Client and content identifiers are grouping keys, not model features.

The following checks examine the feature list, time boundaries, and validation design.

This is a structural audit. It does not independently verify every upstream SQL transformation or re-check the underlying train/test client IDs.

In [ ]:
# --------------------------------------------------
# 1. Check model features
# --------------------------------------------------

feature_names = set(metrics["features"])

forbidden = {
    "click_change_pct",
    "decline_label",
    "late_clicks",
    "late_impressions",
    "late_clicks_per_day",
    "trend_pct",
    "trend_direction",
    "is_declining_label",
    "health_score",
    "priority_score",
    "action_type",
    "client_hash_id",
    "content_hash_id"
}

leakage_hits = sorted(feature_names.intersection(forbidden))

assert not leakage_hits, (
    f"Forbidden features detected: {leakage_hits}"
)

# --------------------------------------------------
# 2. Check feature and outcome chronology
# --------------------------------------------------

feature_end = metrics["feature_window"][1]
outcome_start = metrics["outcome_window"][0]

assert feature_end < outcome_start, (
    "Feature window overlaps with outcome window."
)

# --------------------------------------------------
# 3. Check validation split design
# --------------------------------------------------

assert metrics["split"] == (
    "GroupShuffleSplit by client_hash_id"
)

# --------------------------------------------------
# 4. Build audit report
# --------------------------------------------------

audit = pd.DataFrame([
    [
        "Feature/outcome chronology",
        "PASS",
        "Features end before outcome begins."
    ],
    [
        "Label-derived features",
        "PASS",
        "No forbidden target-derived features found."
    ],
    [
        "Product flags/scores",
        "PASS",
        "Not included in the Week-5 feature list."
    ],
    [
        "Client grouping design",
        "PASS",
        "Week-5 receipt records GroupShuffleSplit by client_hash_id; "
        "underlying IDs are not independently rechecked here."
    ],
    [
        "Base rate",
        "PASS",
        "Base rate is included in model evaluation."
    ],
    [
        "Out-of-sample evaluation",
        "PASS",
        "Week-5 reports metrics on its grouped-client test partition."
    ]
], columns=["check", "status", "evidence"])

print("DATA LEAKAGE AUDIT")
display(audit)

audit.to_csv(
    OUTPUT_DIR / "w06_leakage_audit.csv",
    index=False
)

print("Leakage audit saved successfully.")

## 4. Rewrite Claims to Match the Evidence

### Acceptable claim

"In this March 2026 evaluation, Logistic Regression achieved a Precision@20 of 0.70 on clients held out from training when identifying pages associated with the decline proxy."

### Claims that are NOT supported

1. "The model predicts content decay for all FlyRank clients." The validation covers one within-month grouped-client split.
2. "Refreshing these pages will recover clicks." No intervention or causal experiment was performed.

### Important limitations

- The label is a proxy based on a two-half comparison.
- Results can be affected by seasonality.
- Query mix may change between measurement periods.
- Search Engine Results Page (SERP) changes can affect clicks.
- Tracking availability may be incomplete.
- Small denominators can produce unstable percentage changes.
- The evaluation covers one March period rather than multiple future periods.

### Interpretation

According to the saved Week-5 comparison, the machine learning models outperform the Week-4 rule baseline on several reported ranking metrics. Further validation across multiple periods and unseen clients is required before making broader deployment claims.

## 5. Failure Examples

Week 5 exported held-out predictions to:

`work/outputs/w05_test_predictions.csv`

We will inspect:

- Three false positives.
- Three false negatives.

### False positive

The model predicts that a page will experience a decline, but the actual label indicates no decline.

### False negative

The model predicts that a page will not experience a decline, but the actual label indicates a decline.

Potential explanations include:

- Small denominators.
- Partial measurement coverage.
- Seasonality.
- Query-mix changes.
- Position changes.
- SERP changes.

Do not publish raw identifiers or page-level records in the public report.

In [ ]:
# --------------------------------------------------
# 1. Locate predictions file
# --------------------------------------------------

pred_path = OUTPUT_DIR / "w05_test_predictions.csv"

assert pred_path.exists(), (
    "w05_test_predictions.csv was not found. "
    "Upload it into work/outputs."
)

pred = pd.read_csv(pred_path)

# --------------------------------------------------
# 2. Validate required columns
# --------------------------------------------------

required_columns = {
    "decline_label",
    "logistic_score",
    "click_change_pct"
}

missing_columns = required_columns - set(pred.columns)

assert not missing_columns, (
    f"Missing prediction columns: {missing_columns}"
)

# --------------------------------------------------
# 3. Generate predictions at threshold 0.5
# --------------------------------------------------

pred["predicted_decline"] = (
    pred["logistic_score"] >= 0.5
).astype(int)

# --------------------------------------------------
# 4. Identify false positives
# --------------------------------------------------

fp = (
    pred[
        (pred["decline_label"] == 0) &
        (pred["predicted_decline"] == 1)
    ]
    .head(3)
    .assign(error="false positive")
)

# --------------------------------------------------
# 5. Identify false negatives
# --------------------------------------------------

fn = (
    pred[
        (pred["decline_label"] == 1) &
        (pred["predicted_decline"] == 0)
    ]
    .head(3)
    .assign(error="false negative")
)

# --------------------------------------------------
# 6. Combine and display examples
# --------------------------------------------------

examples = pd.concat([fp, fn], ignore_index=True)

print("FALSE POSITIVE AND FALSE NEGATIVE EXAMPLES")

display(
    examples[
        [
            "error",
            "decline_label",
            "logistic_score",
            "click_change_pct"
        ]
    ]
)

print("\nNumber of false positives:", len(fp))
print("Number of false negatives:", len(fn))

print(
    "\nReview each example and add a human-written "
    "explanation of why the prediction may have been difficult."
)

## 6. Week-6 Conclusion

### What we validated

1. The Week-5 model was evaluated using a grouped-client holdout.
2. The feature list was checked for target-derived and forbidden columns.
3. Feature and outcome windows were checked for chronological separation.
4. The model's performance was compared with the Week-4 rule baseline using the saved Week-5 metrics.
5. False-positive and false-negative examples were prepared for review.

### Main findings

According to the saved Week-5 results:

- Logistic Regression achieved Precision@20 of 0.70, compared with 0.50 for the Week-4 rule baseline.
- Random Forest achieved the highest Average Precision (0.5793) and ROC-AUC (0.5414) among the three methods.
- Logistic Regression achieved Precision@50 of 0.78.

### Limitations

The random-row split diagnostic was not retained in the Week-5 metrics file, so a numerical random-versus-grouped comparison could not be completed from the available outputs.

The evaluation uses a proxy label from one March period. It does not establish causality or guarantee future performance.

### Final conclusion

The Week-5 models demonstrate promising ranking performance for identifying pages associated with the decline proxy.

Further temporal validation, additional client holdouts, and investigation of prediction errors are required before considering deployment.

**Before submission:** complete the two research-paper findings in Section 1 and add human explanations for the displayed failure examples.

In [ ]:
# --------------------------------------------------
# 7. Final Week-6 output check
# --------------------------------------------------

expected_outputs = [
    OUTPUT_DIR / "w06_grouped_results.csv",
    OUTPUT_DIR / "w06_leakage_audit.csv"
]

# Split comparison is optional because the random diagnostic
# was not saved in the current Week-5 metrics receipt.
if random_diag is not None:
    expected_outputs.append(
        OUTPUT_DIR / "w06_split_comparison.csv"
    )

print("WEEK-6 OUTPUT CHECK")

for path in expected_outputs:
    if path.exists():
        print("[OK]", path.name)
    else:
        print("[MISSING]", path.name)

print("\nWeek-6 validation audit completed.")
print(
    "Complete the research-paper findings and human failure-case "
    "annotations before submitting."
)